# Clustering hierárquico

**Objetivo:** construir e ler um **dendrograma** do Iris (ligação de Ward), cortá-lo em $k$ grupos com o `AgglomerativeClustering` e comparar critérios de ligação.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
# amostra pequena para um dendrograma legivel
rng = np.random.RandomState(SEMENTE)
indices = rng.choice(len(X), 30, replace=False)
X_amostra = X[indices]
especies = iris.target_names[iris.target[indices]]
print("amostra:", X_amostra.shape)

## 1. O dendrograma (ligação de Ward)

Cada folha é uma flor; a **altura** de cada união mede o quão diferentes eram os grupos fundidos. Saltos grandes de altura marcam separações naturais — bons lugares para cortar.

In [ ]:
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import linkage

figura = ff.create_dendrogram(X_amostra, labels=list(especies),
                              linkagefun=lambda d: linkage(d, "ward"))
figura.update_layout(title="Dendrograma do Iris (ligacao de Ward)",
                     height=420, margin=dict(l=10, r=10, t=50, b=80))
figura.show()

## 2. Cortando em k grupos

O `AgglomerativeClustering` corta a árvore para dar exatamente $k$ grupos. Com $k=3$, comparamos com as espécies (usando a base completa).

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agrupador = AgglomerativeClustering(n_clusters=3, linkage="ward")
grupos = agrupador.fit_predict(X)
tabela = pd.crosstab(pd.Series(iris.target_names[iris.target], name="especie"),
                     pd.Series(grupos, name="grupo (Ward)"))
print(tabela)

## 3. O critério de ligação muda tudo

Comparamos quatro critérios pelo quanto os grupos batem com as espécies (índice de Rand ajustado, de 0 a 1). A ligação **simples** costuma sofrer com o encadeamento; **Ward** e **completa** vão melhor.

In [ ]:
from sklearn.metrics import adjusted_rand_score

for ligacao in ["single", "complete", "average", "ward"]:
    g = AgglomerativeClustering(n_clusters=3, linkage=ligacao).fit_predict(X)
    ari = adjusted_rand_score(iris.target, g)
    print("ligacao", ligacao.ljust(9), "-> ARI com as especies:", round(ari, 3))

## Exercício

No item 3, a ligação `single` costuma dar o pior ARI. Relacione isso com o efeito de **encadeamento** discutido no texto.

<details><summary>Ver resposta</summary>

A ligação simples define a distância entre grupos pelos **dois pontos mais próximos**. Como *versicolor* e *virginica* se tocam (há pontos de um bem perto do outro), esses pares próximos formam uma "ponte" e o algoritmo funde os dois cedo — o **encadeamento** —, produzindo grupos que não correspondem às espécies e, por isso, um ARI baixo. Ward e completa olham o grupo como um todo e resistem melhor a essa ponte.

</details>